# Projekt numerika

**Autor:** Marek Slavík 



## Ukol čislo 1 

Naprogramujte v programovacím jazyce Python LUPQ rozklad matice $A ∈ R^{n×n}$ s úplnou

pivotizací, tzn. PAQ = LU, například pomocí následujícího pseudokódu:
```
for k = 1 : n − 1
    Determine µ with k ≤ µ ≤ n and λ with k ≤ λ ≤ n
    so |A(µ, λ)| = max{|A(i, j)| such that i = k : n, j = k : n}
    A(k, 1 : n) ↔ A(µ, 1 : n)
    A(1 : n, k) ↔ A(1 : n, λ)
    p(k) = µ
    g(k) = λ
    if A(k, k) ̸= 0
        rows = (k + 1) : n
        A(rows, k) = A(rows, k)/A(k, k)
        A(rows, rows) = A(rows, rows) − A(rows, k) · A(k, rows)
    end if
end for
```


## MOJE IMPLEMENTACE

In [6]:
import numpy as np
import scipy
from scipy import linalg
import matplotlib.pyplot as plt

In [7]:
def paq(A, n):
    
    # Inicializace polí pro uložení indexů výměn
    p = np.arange(n)
    q = np.arange(n)
    

    for k in range(n - 1):
        # sub_A je matice ve které hledame pivot, začíná na pozici (k, k) a zahrnuje všechny prvky pod a vpravo od této pozice
        sub_A = np.abs(A[k:n, k:n]) 
        
        # np.argmax najde 1D pozici největšího prvku, 
        # np.unravel_index ji převede zpět na 2D souřadnice (řádek, sloupec)
        local_mu, local_lam = np.unravel_index(np.argmax(sub_A), sub_A.shape)
        
        # Přepočet lokálních souřadnic na globální (posuneme je o 'k')
        mu = local_mu + k
        lam = local_lam + k
        
        A[[k, mu], :] = A[[mu, k], :]
        p[[k, mu]] = p[[mu, k]]
        
        # 3. Výměna sloupců (k ↔ λ)
        A[:, [k, lam]] = A[:, [lam, k]]
        q[[k, lam]] = q[[lam, k]]
        
        
        # 5. Gaussova eliminace (pokud pivot není nula)
        if A[k, k] != 0:
            rows = slice(k + 1, n)
            # Výpočet matice L
            A[rows, k] = A[rows, k] / A[k, k]
            # Výpočet matice U
            A[rows, rows] = A[rows, rows] - np.outer(A[rows, k], A[k, rows])
            
    return A, p, q

In [8]:
n = 5 
A = np.random.rand(n, n)

a_vysledna, p, g = paq(A, n)
print("Výsledná matice A (obsahuje matici L pod diagonálou a U na a nad diagonálou):")
print(np.round(a_vysledna, 2))
print("\nŘádkové výměny p:", p)
print("Sloupcové výměny g:", g)

Výsledná matice A (obsahuje matici L pod diagonálou a U na a nad diagonálou):
[[ 0.94  0.18  0.71  0.6   0.46]
 [ 0.95  0.71  0.16 -0.29  0.04]
 [ 0.98  0.14 -0.46 -0.38  0.11]
 [ 0.76  0.72  0.98  0.58  0.01]
 [ 0.34  0.97 -0.75  0.2  -0.04]]

Řádkové výměny p: [0 2 3 4 1]
Sloupcové výměny g: [1 2 4 0 3]


## Ukol čislo 4 

Upravte v programovacím jazyce Python implementaci mocninné metody uvedenou na
přednáškách, resp. na cvičení, tak, aby fungovala pro nalezení nejmenšího vlastního čísla
$λ_1$ SPD matice A. Použijte myšlenku o posunu spektra - tzn. hledejte dominantní vlastní
číslo σn matice B = A − $λ_n I$ , kde $λ_n$ je největší vlastní číslo matice A. Pak $λ_1 = σn + λ_n$
(pozor, zde $σ_n = λ_1 − λ_n < 0$). 

In [28]:
def power_method_nejmensivlastnicislo(A, x0, max_it=100, tol=1e-10, use_residual_stop=True):
    A_inv = np.linalg.inv(A)
    
    # vypočtěte počáteční hodnotu q:
    q = x0 / np.linalg.norm(x0)

    for k in range(1, max_it + 1):
        # vypočtěte x:
        x = A_inv @ q
        # vypočtěte novou hodnotu q:
        q = x / np.linalg.norm(x)
        # vypočtěte hodnotu lambda:
        lambda_k = q.T @ A @ q

        if use_residual_stop:
            residual = np.linalg.norm(A @ q - lambda_k * q)
            if residual < tol:
                break

    return lambda_k, q, k

In [ ]:
n= 10
A = np.random.rand(n, n)
A = A @ A.T  # SPD matice (symetrická a pozitivně definitní)

x0 = np.random.rand(n)

# Spuštění metody
lambda_approx, _, iters = power_method_nejmensivlastnicislo(A, x0, max_it=100)

print(f"Aproximované nejmenší vlastní číslo: {lambda_approx:.6f}")
print(f"Počet iterací: {iters}")


# Ověření přes zabudovanou funkci NumPy
skutecna_vlastni_cisla = np.linalg.eigvals(A)
skutecne_nejmensi = np.min(skutecna_vlastni_cisla)

print(f"Skutečné nejmenší vl. číslo (NumPy):    {skutecne_nejmensi:.6f}")

Aproximované nejmenší vlastní číslo: 0.000291
Počet iterací: 6
Skutečné nejmenší vl. číslo (NumPy):    0.000291
